## Problem Statement

### Business Context

As organizations grow, business analysts increasingly face the challenge of navigating large volumes of reports, research papers, and strategic documents. Extracting the right insights from lengthy materials can be time-consuming and overwhelming, especially when these insights directly influence key business decisions.

Consider joining a venture capital firm like Andreessen Horowitz and being assigned a dense report such as Harvard Business Review’s **“How Apple is Organized for Innovation.”** Manually reviewing such documents requires significant effort, slowing down the analysis process and increasing the chances of missing important details.

To overcome this information overload, businesses can leverage **Semantic Search** and **Retrieval-Augmented Generation (RAG)** models. These systems allow analysts to ask natural-language questions like, *“How does Apple structure its teams for innovation?”* and instantly retrieve relevant, accurate insights from the source document.

By integrating such AI-driven retrieval systems, organizations can streamline research workflows, reduce manual effort, and enable analysts to focus on high-value strategic thinking, ultimately improving decision-making speed and quality.


**Common Questions to Answer**

1. Who are the authors of this article and who published this article?

2. List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

3. Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?



### Objective

As an AI specialist, your task is to develop a RAG-based application that enables business analysts to efficiently extract insights from extensive business reports such as “How Apple is Organized for Innovation.” The objective is to understand the challenges of navigating long, information-dense documents, apply retrieval-augmented generation techniques to surface only the most relevant content, analyze how this improves the speed and accuracy of report interpretation, evaluate its potential to enhance strategic decision-making and productivity for analysts, and create a functional prototype that demonstrates the system’s effectiveness in answering queries, summarizing key insights, and supporting natural-language interactions without requiring users to read the entire report.

### Data Description

**“How Apple is Organized for Innovation”** is a detailed Harvard Business Review article that examines Apple’s unique approach to structuring teams, driving innovation, and maintaining a culture of excellence. The article is provided as a PDF consisting of **11 pages**, offering in-depth insights into Apple’s organizational design, leadership principles, and decision-making processes.


## **Please read the instructions carefully before starting the project.**

This is a commented Python Notebook file in which all the instructions and tasks to be performed are mentioned.
* Blanks '_____' are provided in the notebook that
needs to be filled with an appropriate code to get the correct result. With every '_____' blank, there is a comment that briefly describes what needs to be filled in the blank space.
* Identify the task to be performed correctly, and only then proceed to write the required code.
* Please run the codes in a sequential manner from the beginning to avoid any unnecessary errors.
* Add the results/observations (wherever mentioned) derived from the analysis in the presentation and submit the same. Any mathematical or computational details which are a graded part of the project can be included in the Appendix section of the presentation.

## Installing and Importing Necessary Libraries and Dependencies

In [39]:
# Install required libraries
!pip install -q langchain_community==0.3.27 \
              langchain==0.3.27 \
              chromadb==1.0.15 \
              pymupdf==1.26.3 \
              tiktoken==0.9.0 \
              datasets==4.0.0 \
              evaluate==0.4.5 \
              langchain_openai==0.3.30

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [40]:
# Import core libraries
import os                                                                       # Interact with the operating system (e.g., set environment variables)
import json                                                                     # Read/write JSON data
import requests  # type: ignore                                                 # Make HTTP requests (e.g., API calls); ignore type checker

# Import libraries for working with PDFs and OpenAI
from langchain_community.document_loaders import PyMuPDFLoader     # Load and extract text from PDF files
# from langchain_community.document_loaders import PyPDFLoader                    # Load and extract text from PDF files
from openai import OpenAI                                                       # Access OpenAI's models and services

# Import libraries for processing dataframes and text
import tiktoken                                                                 # Tokenizer used for counting and splitting text for models
import pandas as pd                                                             # Load, manipulate, and analyze tabular data

# Import LangChain components for data loading, chunking, embedding, and vector DBs
from langchain.text_splitter import RecursiveCharacterTextSplitter              # Break text into overlapping chunks for processing
from langchain_openai import OpenAIEmbeddings                        # Create vector embeddings using OpenAI's models  # type: ignore
from langchain_community.vectorstores import Chroma                                       # Store and search vector embeddings using Chroma DB  # type: ignore

from datasets import Dataset                                                    # Used to structure the input (questions, answers, contexts etc.) in tabular format
from langchain_openai import ChatOpenAI                                         # This is needed since LLM is used in metric computation

## Question Answering using LLM

### OpenAI API Calling



> **Note:** <br> Make sure to create a `config.json` file in your project directory containing your OpenAI credentials in the following format:
<br><br>```{"API_KEY": "your_openai_api_key_here","OPENAI_API_BASE": "your_api_base"}```<br><br>
Replace the placeholder with your actual API key. This file allows your script to securely load API configuration details without hardcoding them directly into the code. </br>


In [41]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
import json, os

drive_config_path = "/content/drive/MyDrive/Agentic AI Folder/config.json"

config = {
    "API_KEY": "YOUR_OPENAI_API_KEY",
    "OPENAI_API_BASE": "https://api.openai.com/v1"
}

with open(drive_config_path, "w") as f:
    json.dump(config, f)

print("Saved config.json to:", drive_config_path)




Saved config.json to: /content/drive/MyDrive/Agentic AI Folder/config.json


### Response

In [43]:
# Define a function to get a response from the OpenAI chat model
def response(user_prompt, max_tokens=300, temperature=0.7, top_p=1.0):
    """
    Generates a response from the OpenAI chat model.

    Parameters:
    - user_prompt (str): The input text/question from the user.
    - max_tokens (int): Maximum number of tokens the model can generate in its reply.
    - temperature (float): Controls randomness (0 = deterministic, 1 = more creative).
    - top_p (float): Controls diversity using nucleus sampling (1.0 = consider all tokens).

    Returns:
    - str: The generated response text.
    """

    # Create a chat completion request using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o",   # Efficient, fast, and cost-effective OpenAI model
        messages=[
            {"role": "user", "content": user_prompt}  # The user's input message
        ],
        max_tokens=max_tokens,     # Limit how long the response can be
        temperature=temperature,   # Adjust creativity/randomness
        top_p=top_p                # Adjust probability mass sampling
    )

    # Return only the text content of the model's reply
    return completion.choices[0].message.content


In [44]:
import os

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

from openai import OpenAI
client = OpenAI()

print("Client initialized successfully!")


Client initialized successfully!


In [45]:
from openai import OpenAI
client = OpenAI()

print("Client initialized successfully!")


Client initialized successfully!


### Question 1: Who are the authors of this article and who published this article?

In [46]:
question_1 = "Who are the authors of this article and who published this article ?"
base_prompt_response_1=response(question_1)
print(base_prompt_response_1)

I'm sorry, but you haven't provided the article in question. Please provide more information or the text of the article so I can help you identify the authors and the publisher.


### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [47]:
# Define question 2
question_2 = "List down the three leadership characteristics in bulleted points and explain each one under two lines."

# Get response
base_prompt_response_2 = response(question_2)

# Print the output
print(base_prompt_response_2)



- **Empathy**  
  Empathy involves understanding and sharing the feelings of others, which helps leaders connect with their team on a personal level. It fosters a supportive work environment where team members feel valued and heard.

- **Decisiveness**  
  Decisiveness is the ability to make clear, timely decisions even in the face of uncertainty. It helps in maintaining momentum within a team and ensures that goals are pursued with confidence and clarity.

- **Integrity**  
  Integrity involves adhering to strong moral and ethical principles, promoting trust and respect within the team. Leaders who exhibit integrity are consistent in their actions and decisions, which builds credibility and reliability.


### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [48]:
# Define question 3
question_3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"

# Get response
base_prompt_response_3 = response(question_3)

# Print output
print(base_prompt_response_3)



I'm sorry, but I don't have access to specific articles or their contents. However, I can provide general information about Apple's approach to leadership and how it has led to successful innovations.

Apple's leadership, particularly under Steve Jobs and now Tim Cook, has been characterized by a few key principles that have driven innovation:

1. **Focus on Design and User Experience**: Apple's leadership has always placed a strong emphasis on design and the user experience. This focus has led to the creation of products like the iPhone, iPad, and MacBook, which are known for their sleek design and intuitive interfaces.

2. **Integration of Hardware and Software**: Apple’s strategy of tightly integrating its hardware and software ecosystems has resulted in seamless user experiences. Innovations like the A-series chips in iPhones and the M1 chips in Macs are examples of this approach, leading to significant performance and efficiency gains.

3. **Culture of Secrecy and Surprise**: Appl

**Observations:**
- When I ask general question like:List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines, it provided an output
-On specific questions, I did not get an output. Instead I got: I'm sorry, but I can't access external content or databases to find specific articles or their authors. If you provide me with the title of the article or more context, I might be able to help you with general information or guidance on how to find that information.

## Question Answering using LLM with Prompt Engineering

In the next step, we will use prompt engineering to check the effect of a more detailed and well-engineered prompt on the output of the model.

In [49]:
system_prompt = """
You are a helpful AI assistant specialized in question answering.

Provide clear, structured, and concise answers.
If the question asks for bullet points, format the response in bullet points.
Keep explanations brief (no more than two lines per point).
Do not fabricate information. If information is missing, clearly state that.
"""

### Defining the function to Generate a Response From the LLM

In [50]:
# Define a function to get a response from the OpenAI chat model
def response(system_prompt, user_prompt, max_tokens=300, temperature=0.7, top_p=1.0):
    """
    Generates a response from the OpenAI chat model using system and user prompts.

    Parameters:
    - system_prompt (str): Instructions that define the assistant's behavior.
    - user_prompt (str): The user's input question or request.
    - max_tokens (int): Maximum number of tokens to generate.
    - temperature (float): Controls randomness (0 = deterministic, 1 = creative).
    - top_p (float): Controls diversity using nucleus sampling.

    Returns:
    - str: The model's response text.
    """

    # Create a chat completion request using the OpenAI client
    completion = client.chat.completions.create(
        model="gpt-4o",  # Efficient and cost-effective model for general tasks
        messages=[
            {"role": "system", "content": system_prompt},  # Defines assistant behavior
            {"role": "user", "content": user_prompt}       # Actual user question
        ],
        max_tokens=max_tokens,     # Limit response length
        temperature=temperature,   # Control randomness
        top_p=top_p                # Control diversity
    )

    # Return the generated response text
    return completion.choices[0].message.content


### Question 1: Who are the authors of this article and who published this article ?

In [51]:
response_with_prompt_eng_1=response(system_prompt,question_1)
response_with_prompt_eng_1

"I'm sorry, but I need more context or a specific article title to identify the authors and publisher. Please provide more details or the article's name."

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [52]:
response_with_prompt_eng_2 = response(system_prompt, question_2)
response_with_prompt_eng_2


'- **Visionary**: A leader with a clear vision can inspire and motivate others by setting a compelling and achievable direction for the future.\n\n- **Decisive**: Effective leaders make timely decisions, considering all relevant information and potential outcomes, to guide their teams with confidence.\n\n- **Empathetic**: Demonstrating empathy allows leaders to understand and connect with their team members, fostering trust and a supportive work environment.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [53]:
response_with_prompt_eng_3 = response(system_prompt, question_3)
response_with_prompt_eng_3

"I'm sorry, but I don't have access to specific articles. However, I can provide general examples of Apple's leadership approach leading to successful innovations:\n\n- **Focus on Design**: Apple's emphasis on minimalist and user-friendly design, led by Steve Jobs, resulted in iconic products like the iPhone and MacBook.\n\n- **Vertical Integration**: Under Tim Cook, Apple's control over its supply chain and hardware-software integration has ensured high-quality products and seamless user experiences.\n\n- **Innovation Culture**: Apple's leadership fosters a culture of innovation, encouraging teams to push the boundaries, as seen with the development of the Apple Watch and AirPods. \n\nIf you need details from a specific article, please provide more context or excerpts."

**Observations**:
-Question 2 (Leadership traits): The model successfully produced a well-structured and concise response because the question was based on general knowledge. Prompt engineering improved formatting and clarity, but the answer was not grounded in any specific article.

Question 3 (Apple examples): The model could not provide article-specific examples because no document context was supplied. This demonstrates that prompt engineering controls behavior and structure, but retrieval (RAG) is required for fact-based, document-grounded answers.
-


## Data Preparation for RAG

### Loading the Data

In [54]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
!ls /content/drive

MyDrive


In [ ]:
!ls /content/drive/MyDrive

In [56]:
pdf_path = "/content/drive/MyDrive/Agentic AI Folder/HBR_How_Apple_Is_Organized_For_Innovation.pdf"

In [57]:
pdf_loader = PyMuPDFLoader(pdf_path)

In [58]:
pdf = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(pdf[i].page_content,end="\n")

### Data Chunking

#### Chunk the PDF into Manageable Text Sections Using a Token-Based Splitter

In [60]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,       # Number of tokens per chunk
    chunk_overlap=200      # Overlap between chunks to preserve context
)

#### Split the Loaded PDF into Chunks for Further Processing

In [61]:
document_chunks = pdf_loader.load_and_split(text_splitter)

#### Check the Number of Chunks Created

In [62]:
len(document_chunks)

17

In [63]:
import os
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings

os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

client = OpenAI()

embedding_model = OpenAIEmbeddings()

print("Environment restored successfully.")


Environment restored successfully.


In [64]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
print("Dimension:", len(embedding_1))


Dimension: 1536


### Generate Vector Embeddings for Text Chunks Using OpenAI

In [66]:
# Initialize the OpenAI Embeddings model with API credentials
embedding_model = OpenAIEmbeddings()

# Generate embeddings (vector representations) for the first two document chunks
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)      # Embedding for chunk 0
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)      # Embedding for chunk 1

# Check and print the dimension (length) of the embedding vector
print("Dimension of the embedding vector ", len(embedding_1))                   # Typically 1536 or 2048 depending on model

Dimension of the embedding vector  1536


In [ ]:
# Verify if both embeddings have the same dimension (should be True)
len(embedding_1) == len(embedding_2)

# Return/display the two embedding vectors for further inspection or use
embedding_1, embedding_2

### Vector Database

#### Setup Vector Database Directory

In [68]:
out_dir = 'vector_db'    # complete the code to define the name of the vector database

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

#### Create Vector Database from Documents

In [69]:
# Building the vector store and saving it to disk for future use
vectorstore = Chroma.from_documents(
    document_chunks,                                                            # Documents to index
    embedding_model,                                                            # Embedding model for converting text to vectors
    persist_directory=out_dir                                                   # Save vector DB files here
)

#### Load Vector Database

In [70]:
vectorstore = Chroma(
    persist_directory=out_dir,
    embedding_function=embedding_model
)

/tmp/ipython-input-3612695989.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(


#### Explore Vector Database and Perform Searches

In [71]:
vectorstore.embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x787e1cb699d0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x787e1cb927b0>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

write an instruction on what to do in next cell

In [ ]:
vectorstore.similarity_search("What leadership structure does Apple use to drive innovation?", k=3) ## Converts the query into a vector. Compares it to your 17 stored chunk vectors. Returns the top k most similar chunks

### Retrieval and Response Generation using Vector Search

#### Convert Vector Database into a Retriever and Retrieve Relevant Documents

In [73]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}  # Retrieve top 3 most relevant chunks
)

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

    1. The system message describing the assistant's role.
    2. A user message template including context and the question.

In [74]:
qna_system_message = """
You are a helpful AI assistant that answers questions based only on the provided context.

Use the retrieved document excerpts to generate your answer.
If the answer is not found in the context, clearly state that the information is not available.
Be concise, accurate, and avoid adding information that is not supported by the context.
"""

In [75]:
qna_user_message_template = """
Context:
{context}

Question:
{question}

Answer:
"""


### Response Function

In [81]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    # Generate the response
    try:
        response = client.chat.completions.create(
        model="gpt-4o",   # Complete the code by specifying the model to be used.
        messages=[
            {"role": "system", "content": qna_system_message},
            {"role": "user", "content": user_message}
        ],
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p
        )
        # Extract and print the generated text from the response
        response = response.choices[0].message.content.strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Question 1: Who are the authors of this article and who published this article ?

In [84]:
response_with_rag_1 = generate_rag_response("Who are the authors of this article and who published this article ?")
response_with_rag_1

'The authors of the article are Joel M. Podolny and Morten T. Hansen. The article was published in the Harvard Business Review.'

### Question 2: List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines.

In [82]:
response_with_rag_2 = generate_rag_response("List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines") #Complete the code to pass the user input
response_with_rag_2

'- **Deep Expertise**: Leaders at Apple are expected to have profound knowledge in their specific areas, allowing them to engage meaningfully in the work and make informed decisions.\n\n- **Immersion in Details**: Apple leaders are deeply involved in the specifics of their functions, enabling them to identify critical issues and focus their attention effectively.\n\n- **Willingness to Collaboratively Debate**: Leaders engage in open discussions with colleagues from various functions, promoting or challenging ideas to achieve the best solutions collectively.'

### Question 3: Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?

In [83]:
response_with_rag_3 = generate_rag_response("Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations") #Complete the code to pass the user input
response_with_rag_3

"The article provides specific examples of Apple's approach to leadership leading to successful innovations:\n\n1. **Roger Rosner and Software Applications**: Roger Rosner, who leads Apple's software application business, exemplifies Apple's approach of having experts lead experts. His deep expertise in engineering and software development has contributed to the success of Apple's productivity apps like Pages, Numbers, and Keynote, as well as creative apps like GarageBand and iMovie.\n\n2. **Graham Townsend and Camera Technology**: Graham Townsend leads a group of over 600 experts in camera hardware technology. This centralized expertise allows for effective problem-solving and innovation across Apple's product lines, such as"

### **Observations**

-Answers become article-grounded (less guessing): Before RAG, the model either refused (“I don’t have the article”) or gave generic leadership traits. With RAG, it pulls the most relevant chunks from the PDF first, so it can cite/reflect what the article actually says (e.g., authors/publisher details and Apple’s organizational/leadership model) instead of inventing.

Better handling of “this article” questions: Without RAG, questions like “Who are the authors/publisher?” and “Give examples from the article…” fail because the model has no access to the document. With RAG, those become answerable because the retriever supplies the needed context right into the prompt.

Quality shifts from fluent → accurate & verifiable: Without RAG, outputs may sound confident but can be wrong (hallucination-by-generalization). With RAG, you can inspect the retrieved chunks (via similarity_search/retriever) to verify the evidence behind the answer, and the response is constrained by what’s in the document.



## Actionable Insights and Business Recommendations


* Actionable Insight 1: RAG Significantly Improves Answer Accuracy for Document-Based Tasks Business recommendation 1: Organizations deploying AI for internal knowledge management should implement RAG architectures to ensure responses are grounded in company documents, policies, and reports rather than relying solely on general LLM knowledge.
* Actionable Insight 2: RAG Reduces Hallucination Risk and Increases Trust. Business recommendation 2:For high-stakes use cases (legal, healthcare, finance, corporate reporting), companies should adopt RAG systems with retrieval visibility to enhance auditability and reduce misinformation risk.  
*Actionable Insight 3: RAG Increases System Complexity but Improves Strategic Capability Business recommendation 3:Organizations should invest in AI engineering capabilities (vector databases, retrieval pipelines, prompt governance) rather than relying solely on prompt-based experimentation for enterprise-grade AI deployment.



<font size=6 color='#4682B4'>Power Ahead</font>
___